# 🏦 FinTrust Digital Bank — Part C: Python Exploratory Data Analysis (EDA)

**Track:** Data Analytics Track  
**Project:** FinTrust Customer & Transaction Analytics  
**Deliverable File:** `FinTrust_Week2_Data_Analysis.ipynb`  
**Tools Required:** Python, Pandas, NumPy, Matplotlib, Seaborn  

---

## 📌 Notebook Overview
This notebook executes a comprehensive exploratory data analysis (EDA) on FinTrust Digital Bank's customer (`FinTrust_Data_customer.csv`) and transaction (`FinTrust_Data_transaction.csv`) datasets. It investigates customer segmentation, transaction channels, monetary values, failure rates, international transfers, and risk-review patterns, concluding with 5 key business visualizations and strategic recommendations.

In [ ]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print("Libraries imported successfully.")

In [ ]:
# Step 2: Load Customer and Transaction Datasets
cust_df = pd.read_csv("FinTrust_Data_customer.csv")
tx_df = pd.read_csv("FinTrust_Data_transaction.csv")

# Convert Transaction_DateTime to native pandas datetime
tx_df['Transaction_DateTime'] = pd.to_datetime(tx_df['Transaction_DateTime'])

# Merge datasets
merged_df = pd.merge(tx_df, cust_df, on="Customer_ID", how="inner", suffixes=('', '_cust'))

print(f"Customer Records: {len(cust_df):,}")
print(f"Transaction Records: {len(tx_df):,}")
print(f"Merged Dataset Shape: {merged_df.shape}")

In [ ]:
# Visualization 1: Total Transaction Value by Customer Segment
fig, ax1 = plt.subplots(figsize=(10, 5))

segment_summary = merged_df.groupby('Customer_Segment_cust').agg(
    Total_Value=('Amount_NGN', lambda x: x.sum() / 1e6),
    Tx_Count=('Transaction_ID', 'count')
).reset_index().sort_values(by='Total_Value', ascending=False)

bars = sns.barplot(data=segment_summary, x='Customer_Segment_cust', y='Total_Value', palette="mako", ax=ax1)
ax1.set_title("1. Financial Liquidity Share by Customer Segment (NGN Millions)", fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel("Customer Segment", fontsize=11, fontweight='bold')
ax1.set_ylabel("Total Transaction Value (₦ Millions)", fontsize=11, fontweight='bold')

for bar in bars.patches:
    height = bar.get_height()
    ax1.annotate(f"₦{height:.2f}M", xy=(bar.get_x() + bar.get_width() / 2, height), xytext=(0, 5),
                 textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### 📊 Visual 1 Analysis & Business Insight
* **What the visual shows:** Total processed financial value (in NGN Millions) grouped across the four bank customer tiers: Everyday, Premium, Student, and SME.
* **Why it is important:** It highlights where FinTrust's core capital and deposit liquidity originate, enabling targeted product pricing and customer acquisition.
* **Business Insight:** The **Everyday** segment commands **46.65% (₦261.46M)** of all processed value. Surprisingly, the **Student** segment matches the total financial output of **Premium** clients (~₦107.48M), signaling strong long-term lifetime value (LTV) potential.

In [ ]:
# Visualization 2: Channel Volume Share and Technical Failure Rates
channel_summary = tx_df.groupby('Channel').agg(
    Total_Tx=('Transaction_ID', 'count'),
    Failed_Tx=('Transaction_Status', lambda x: (x == 'Failed').sum())
).reset_index()

channel_summary['Failure_Rate_%'] = (channel_summary['Failed_Tx'] / channel_summary['Total_Tx']) * 100
channel_summary = channel_summary.sort_values(by='Total_Tx', ascending=False)

fig, ax1 = plt.subplots(figsize=(10, 5))
color = '#1B365D'
ax1.set_xlabel('Payment Channel', fontsize=11, fontweight='bold')
ax1.set_ylabel('Total Transaction Volume', color=color, fontsize=11, fontweight='bold')
bars = ax1.bar(channel_summary['Channel'], channel_summary['Total_Tx'], color=color, alpha=0.85, width=0.5)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#E53E3E'
ax2.set_ylabel('Failure Rate (%)', color=color, fontsize=11, fontweight='bold')
line = ax2.plot(channel_summary['Channel'], channel_summary['Failure_Rate_%'], color=color, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(3.0, 7.0)

plt.title("2. Transaction Volume vs. Technical Failure Rate by Channel", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 📊 Visual 2 Analysis & Business Insight
* **What the visual shows:** Transaction volume per channel alongside its corresponding technical failure rate percentage.
* **Why it is important:** Identifies operational bottlenecks and channel instability that negatively affect user experience and cause revenue leakage.
* **Business Insight:** **Mobile App** handles the vast majority of volume (5,102 transactions, 42.52% share) but suffers from the **highest failure rate (5.80%)**. Mobile API gateway optimization is necessary to prevent customer churn.

In [ ]:
# Visualization 3: Financial Value Share by Transaction Type
type_summary = tx_df.groupby('Transaction_Type')['Amount_NGN'].sum().reset_index()
type_summary['Value_Share'] = (type_summary['Amount_NGN'] / type_summary['Amount_NGN'].sum()) * 100
type_summary = type_summary.sort_values(by='Value_Share', ascending=False)

plt.figure(figsize=(7, 7))
colors = ['#1B365D', '#2B6CB0', '#4299E1', '#63B3ED', '#90CDF4', '#CBD5E0']

plt.pie(
    type_summary['Value_Share'], 
    labels=type_summary['Transaction_Type'],
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    pctdistance=0.80,
    wedgeprops=dict(width=0.4, edgecolor='w', linewidth=2)
)

plt.title("3. Financial Value Share by Transaction Type", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 📊 Visual 3 Analysis & Business Insight
* **What the visual shows:** Donut chart breakdown of capital share across transaction types (Transfers, Deposits, Card Purchases, Withdrawals, Bill Payments, Airtime).
* **Why it is important:** Reveals how customers utilize FinTrust's core banking services.
* **Business Insight:** **Transfers (42.5%)** and **Deposits (23.4%)** make up over **65.8% of total financial volume**. Everyday micro-services (Airtime/Data and Bill Payments) drive transaction frequency rather than monetary value.

In [ ]:
# Visualization 4: Monthly Transaction Value Trend (Q1 2026)
tx_df['Month_Name'] = tx_df['Transaction_DateTime'].dt.strftime('%B')
monthly = tx_df.groupby('Month_Name')['Amount_NGN'].sum().reindex(['January', 'February', 'March']).reset_index()
monthly['Value_Millions'] = monthly['Amount_NGN'] / 1e6

plt.figure(figsize=(10, 4.5))
plt.plot(monthly['Month_Name'], monthly['Value_Millions'], marker='o', linewidth=3, color='#1B365D', markersize=8)
plt.fill_between(monthly['Month_Name'], monthly['Value_Millions'], color='#1B365D', alpha=0.1)

plt.title("4. Q1 2026 Monthly Processed Financial Throughput (₦ Millions)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Month", fontsize=11, fontweight='bold')
plt.ylabel("Total Value (₦ Millions)", fontsize=11, fontweight='bold')
plt.ylim(160, 210)

for x, y in zip(monthly['Month_Name'], monthly['Value_Millions']):
    plt.annotate(f"₦{y:.2f}M", (x, y), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 📊 Visual 4 Analysis & Business Insight
* **What the visual shows:** Month-by-month financial throughput trend across Q1 2026.
* **Why it is important:** Tracks revenue growth momentum and seasonal variations.
* **Business Insight:** Following a brief dip in February (short calendar month), total transaction throughput reached a record peak in **March 2026 at ₦196.20 Million**, reflecting growing customer activity.

In [ ]:
# Visualization 5: Transaction Status Breakdown by Risk Review Flag
risk_status = pd.crosstab(tx_df['Risk_Review_Flag'], tx_df['Transaction_Status'], normalize='index') * 100

ax = risk_status.plot(kind='bar', stacked=True, figsize=(9, 5), color=['#E53E3E', '#DD6B20', '#D69E2E', '#319795'])

plt.title("5. Transaction Completion Status Distribution by Risk Review Flag", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Flagged for Risk Review (Yes / No)", fontsize=11, fontweight='bold')
plt.ylabel("Percentage Share (%)", fontsize=11, fontweight='bold')
plt.legend(title="Transaction Status", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

### 📊 Visual 5 Analysis & Business Insight
* **What the visual shows:** Comparative completion status distribution (Successful, Failed, Reversed, Pending) for transactions flagged vs unflagged for risk review.
* **Why it is important:** Evaluates anti-fraud interception accuracy and operational risk rules.
* **Business Insight:** Flagged transactions exhibit a **higher non-success rate (9.78%)** than unflagged transactions (7.52%), proving that automated risk detection correctly intercepts suspicious payment traffic.